In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchvision.transforms import ToTensor
import torch

device = torch.device("cuda")
train_dataset = ImageFolder("/content/drive/MyDrive/deepL/melanoma_cancer_dataset/train", transform=ToTensor())


test_dataset = ImageFolder("/content/drive/MyDrive/deepL/melanoma_cancer_dataset/test", transform=ToTensor())

train_dataloader = DataLoader(train_dataset, batch_size=100, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=100, shuffle=True)

print(type(train_dataloader))
# Display image and label.
# train_features, train_labels = datset_train[0]
#print(f"Feature batch shape: {train_features.size()}")
#print(f"Labels batch shape: {train_labels.size()}")
#img = train_features[0].squeeze()
#label = train_labels[0]
#plt.imshow(img)

#plt.imshow(img[0])
#print(f"Label: {label}")




<class 'torch.utils.data.dataloader.DataLoader'>


In [3]:
import torch.nn as nn

class AlexNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(

            nn.Conv2d(in_channels = 3, out_channels = 48, kernel_size = (11,11), stride = 4),

            nn.Conv2d(48, 128, (5,5)),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),

            #nn.Conv2d(128, 192, 3),
            #nn.Conv2d(192, 192, 3),
            #nn.Conv2d(192, 128, 3),

            nn.MaxPool2d((2,2)),

            nn.Flatten(),

            nn.Dropout(0.5),
            nn.Linear(36992,2048),
            nn.ReLU(),

            nn.Dropout(0.5),
            nn.Linear(2048,200),
            nn.ReLU(),
            nn.Linear(200, 2)

        )

    def forward(self, X):
        return self.seq(X)

model = AlexNet()
model.to(device)
print(model)

AlexNet(
  (seq): Sequential(
    (0): Conv2d(3, 48, kernel_size=(11, 11), stride=(4, 4))
    (1): Conv2d(48, 128, kernel_size=(5, 5), stride=(1, 1))
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (5): Flatten(start_dim=1, end_dim=-1)
    (6): Dropout(p=0.5, inplace=False)
    (7): Linear(in_features=36992, out_features=2048, bias=True)
    (8): ReLU()
    (9): Dropout(p=0.5, inplace=False)
    (10): Linear(in_features=2048, out_features=200, bias=True)
    (11): ReLU()
    (12): Linear(in_features=200, out_features=2, bias=True)
  )
)


In [4]:

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [5]:
import torch.optim

lr = 0.0005
batch_size = 100



loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.698350  [  100/ 9605]
Test Error: 
 Accuracy: 75.4%, Avg loss: 0.496562 

Epoch 2
-------------------------------
loss: 0.537181  [  100/ 9605]
Test Error: 
 Accuracy: 66.9%, Avg loss: 0.534309 

Epoch 3
-------------------------------
loss: 0.580526  [  100/ 9605]
Test Error: 
 Accuracy: 81.6%, Avg loss: 0.394268 

Epoch 4
-------------------------------
loss: 0.444125  [  100/ 9605]
Test Error: 
 Accuracy: 85.6%, Avg loss: 0.322442 

Epoch 5
-------------------------------
loss: 0.303269  [  100/ 9605]
Test Error: 
 Accuracy: 87.2%, Avg loss: 0.316781 

Epoch 6
-------------------------------
loss: 0.304459  [  100/ 9605]
Test Error: 
 Accuracy: 87.1%, Avg loss: 0.282810 

Epoch 7
-------------------------------
loss: 0.270789  [  100/ 9605]
Test Error: 
 Accuracy: 88.1%, Avg loss: 0.277879 

Epoch 8
-------------------------------
loss: 0.336130  [  100/ 9605]
Test Error: 
 Accuracy: 89.0%, Avg loss: 0.266695 

Epoch 9
----------------

In [6]:
torch.cuda.is_available()

True

In [7]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [8]:
torch.accelerator.current_accelerator()

device(type='cuda')